# 🏆 สมุดบันทึกที่ 4: การเปรียบเทียบประสิทธิภาพและการสรุปผล (ทำร่วมกัน)
**วิชา:** การทำเหมืองข้อมูล (Data Mining)  
**เนื้อหาอ้างอิง:** บทที่ 5.4 (การวัดประสิทธิภาพของ Classification Model) และบทที่ 6.2.7 (เปรียบเทียบโมเดล)  
**ผู้รับผิดชอบ:** ทั้งสองคนร่วมกัน  
**เป้าหมาย:** ประชันผลลัพธ์ของ Decision Tree vs Naive Bayes หมัดต่อหมัด, สรุปข้อดี-ข้อเสีย, และสังเคราะห์องค์ความรู้ (Actionable Insights) เสนอแนะแก่โรงเรียน


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

train_df = pd.read_csv('../data/processed/train.csv')
test_df = pd.read_csv('../data/processed/test.csv')

X_train = train_df.drop(columns=['passed'])
y_train = train_df['passed']
X_test = test_df.drop(columns=['passed'])
y_test = test_df['passed']

# 1. เทรนโมเดลตัวแทนที่ดีที่สุดของทั้งสองคน
dt_model = DecisionTreeClassifier(random_state=42, ccp_alpha=0.00681).fit(X_train, y_train)
nb_model = GaussianNB().fit(X_train, y_train)

print("✅ โหลดข้อมูลและเทรนโมเดลทั้งสองเสร็จสมบูรณ์")


## 1. ตารางเปรียบเทียบประสิทธิภาพหมัดต่อหมัด (Head-to-Head Comparison)


In [ ]:
models = {
    "Decision Tree (Post-Pruned)": dt_model,
    "Gaussian Naive Bayes": nb_model
}

summary = []
for name, m in models.items():
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    
    summary.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (Pass)": precision_score(y_test, pred, pos_label=1),
        "Recall (Pass)": recall_score(y_test, pred, pos_label=1),
        "Precision (Fail)": precision_score(y_test, pred, pos_label=0),
        "Recall (Fail)": recall_score(y_test, pred, pos_label=0),
        "F1-Score (Macro)": f1_score(y_test, pred, average='macro'),
        "ROC-AUC": roc_auc_score(y_test, proba)
    })

comparison_df = pd.DataFrame(summary)
print("=" * 80)
print("📊 ตารางเปรียบเทียบผลลัพธ์ประสิทธิภาพ (Evaluation Comparison Table):")
print("=" * 80)
comparison_df


## 2. เปรียบเทียบ Confusion Matrix วางคู่กัน


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_dt = confusion_matrix(y_test, dt_model.predict(X_test))
cm_nb = confusion_matrix(y_test, nb_model.predict(X_test))

sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Fail (0)', 'Pass (1)'], yticklabels=['Fail (0)', 'Pass (1)'])
axes[0].set_title('Decision Tree (Post-Pruned)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Fail (0)', 'Pass (1)'], yticklabels=['Fail (0)', 'Pass (1)'])
axes[1].set_title('Gaussian Naive Bayes')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../images/confusion_matrices.png', dpi=300)
plt.show()


## 3. พล็อตกราฟเปรียบเทียบเส้น ROC Curves


In [ ]:
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_model.predict_proba(X_test)[:, 1])
fpr_nb, tpr_nb, _ = roc_curve(y_test, nb_model.predict_proba(X_test)[:, 1])

auc_dt = roc_auc_score(y_test, dt_model.predict_proba(X_test)[:, 1])
auc_nb = roc_auc_score(y_test, nb_model.predict_proba(X_test)[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(fpr_dt, tpr_dt, color='#2b5c8f', lw=2, label=f'Decision Tree (AUC = {auc_dt:.4f})')
plt.plot(fpr_nb, tpr_nb, color='#d35400', lw=2, label=f'Gaussian Naive Bayes (AUC = {auc_nb:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Chance (AUC = 0.5)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves Comparison: Decision Tree vs Naive Bayes', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.savefig('../images/roc_curves.png', dpi=300)
plt.show()


## 4. บทสรุปและการอภิปรายผล (Discussion & Actionable Insights)

### 4.1 ข้อดี-ข้อเสีย และความเหมาะสมของแต่ละโมเดล:
* **Decision Tree (ชนะเลิศด้านภาพรวมและความโปร่งใส):**
  * ให้ค่า Accuracy สูงถึง **80.00%** และ F1-Score **0.7294**
  * เหมาะสมกับสถานการณ์ที่ต้องการ **อธิบายเหตุผลเป็นกฎข้อๆ (Explainable AI / White-Box)** แก่อาจารย์และผู้ปกครอง
* **Gaussian Naive Bayes (ชนะเลิศด้านการตรวจจับเด็กกลุ่มเสี่ยง):**
  * มีค่า **Recall ของคลาส Fail สูงถึง 50.00%** (จับเด็กตกได้ 10 คน เทียบกับ Decision Tree ที่จับได้ 5 คน)
  * เหมาะสมกับระบบเตือนภัยล่วงหน้า (Early Warning) ที่ยอมให้มี False Alarm ดีกว่าปล่อยให้นักเรียนที่กำลังจะสอบตกหลุดรอดไป

### 4.2 ข้อเสนอแนะเชิงนโยบายสำหรับโรงเรียน (Actionable Insights):
1. **เฝ้าระวังเด็กที่มีประวัติเคยสอบตกสะสม (`failures >= 1`):** เนื่องจากเป็นปัจจัยที่มีน้ำหนักสูงสุด (58%) โรงเรียนควรมีระบบจับคู่อาจารย์ที่ปรึกษาตั้งแต่สัปดาห์แรกของภาคเรียน
2. **สร้างแรงบันดาลใจทางการศึกษา (`higher`):** นักเรียนที่มีความตั้งใจจะศึกษาต่อระดับมหาวิทยาลัยมีอัตราสอบผ่านสูงกว่าอย่างมีนัยสำคัญ โรงเรียนควรจัดกิจกรรมแนะแนวอาชีพและเป้าหมายในอนาคตเพื่อกระตุ้นความตั้งใจเรียน
